# Full-epoch decoding — one decision per trial, all subjects

Where `Windowed_Analysis.ipynb` asks *"what accuracy at each moment in time"* (the live
question), this asks **"given the whole trial, how well can it be classified"** — a single
cross-validated number per subject, which is what a manuscript reports.

Every step lives in `src/full_epoch_batch.py`; this notebook is only the driver.
**All settings live in the Parameters cell below** — nothing under it needs editing.

| stage | call | writes |
|---|---|---|
| 1 | `run_full_epoch_batch()` | `Analysis/<label>/FullEpoch/Metrics/Individuals/`, `.../Figures/<subject>/` |
| 2 | `summarize_full_epoch()` | `.../FullEpoch/Metrics/Group/`, `.../Figures/Group/`, `.../full_epoch_summary.{csv,json}` |
| 3 | `run_permutation_pass()` | `.../FullEpoch/permutation_<definition>_<mode>.json` — **opt-in, expensive** |

Outputs live under **`Analysis/<label>/FullEpoch/`** — one tree per *class set*
(`MH_LH_RH_FR`, `LH_RH`, ...) — so changing `desired_events` never overwrites a previous run.
The epoch cache at `Analysis/<label>/Cache/` is **shared** with the windowed analysis, so
nothing is re-preprocessed and ~2.4 GB is not duplicated. Code common to the two analyses lives
in `src/analysis_common.py`.

## Two definitions of "full epoch"

| definition | what it does |
|---|---|
| `vote` | train on the classifier-window crop, slide 2 s windows across [0, 5] s, take the modal prediction per trial |
| `single` | train and test on **one** window spanning [0, 5] s — a single covariance per trial, no sliding, no voting |

## Seven centering modes, and why there are seven

Class-mean centering is **label-dependent** — you must know a trial's class to centre it — so
any mode whose *test* source is centered reports something a decoder cannot achieve on
unlabelled data. Three separate things are tangled together, and each delta isolates exactly one:

| recipe | class means computed over |
|---|---|
| `c` (per file) | each XDF file separately — what `EEG_Preprocessing` produces, so it *also* removes between-session drift |
| `g` (global) | every trial of the subject |
| `l` (fold-local) | the fold's **training** trials only — no self-inclusion |
| `u` | not centered |

Modes are `<train>2<test>`: `c2c`, `c2u`, `g2g`, `g2u`, `l2l`, `l2u`, `u2u`. Only `c2c` runs by
default; set `RUN_MODES = list(DIAGNOSTIC_MODES)` in the Parameters cell for the full diagnostic.

| delta | isolates |
|---|---|
| `g2g − l2l` | the self-inclusion leak on the **test** side |
| `g2u − l2u` | the self-inclusion leak on the **train** side |
| `l2l − l2u` | what label-dependent test centering is worth once self-inclusion is gone |
| `c2c − g2g` | what the per-file (session-drift) recipe adds over global |
| `l2u − u2u` | whether training-side centering helps at all, **deployably** |

`c2c` and `g2g` cannot be compared to `l2l` directly without the `g` baseline: file boundaries
are not recoverable from the concatenated epochs, so a per-file recipe cannot be re-estimated
on a different trial subset. That is why `g` exists.

**`l2u` is the only fully leak-free deployable number.**

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import warnings; warnings.filterwarnings('ignore')

from src.full_epoch_batch import *

# src.preprocessing forces %matplotlib qt at import time, so set the backend after it.
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')

print('subjects          :', list_subjects())
print('electrode groups  :', list(ELECTRODE_GROUPS))
print('definitions       :', list(DEFINITIONS))
print('modes             :', list(MODES), '| diagnostic:', list(DIAGNOSTIC_MODES))
print('window            :', FULL_EPOCH_WINDOW)

subjects          : ['AEH', 'AN', 'AV', 'BA', 'EA', 'EE', 'ID', 'IO', 'LD', 'MN', 'NS', 'NZ', 'SK', 'SM', 'SN', 'TM']
electrode groups  : ['FP', 'AF', 'F', 'FC', 'C', 'CP', 'P', 'PO', 'O']
definitions       : ['vote', 'single']
modes             : ['c2c'] | diagnostic: ['c2c', 'c2u', 'g2g', 'g2u', 'l2l', 'l2u', 'u2u']
window            : (0.0, 5.0)


## Parameters — the only cell you need to edit

Everything the pipeline reads, spelled out. `default_params()` seeds it, then each key is set
explicitly so the value is visible and editable here rather than in `src/`.

`check_params` rejects a misspelled key (which would otherwise be silently ignored) and the
combinations that fail deep inside preprocessing; `describe_params` marks with `*` whatever
differs from the library default.

**`LABEL` matters.** The output tree is named from `desired_events` alone, and group results are
merged by subject rather than replaced — so re-running with a different filter band or electrode
set would mix the two runs. `check_params` warns when the tree on disk was built with different
params; set `LABEL` to a new name when it does.

**Definition `single` overrides three keys.** It rewrites `classifier_window_s`,
`classifier_window_e` and `windowed_prediction_params` to span `FULL_EPOCH_WINDOW`, because
"train and test on the whole epoch" is expressed as one window that fits exactly once
(`full_epoch_batch.full_epoch_params`). Those three are `vote`-only knobs.

In [2]:
# ══ PARAMETERS ═══════════════════════════════════════════════════════════════════
ELECTRODE_GROUP_NAMES = 'FC+C+CP+P'   # groups available: printed by the cell above
LABEL = None                          # Analysis/<label>/ tree; None -> named from
                                      # desired_events. Set it when you change any
                                      # other param (see the note above).

PARAMS = default_params(electrode_group_names=ELECTRODE_GROUP_NAMES)

# ── active ───────────────────────────────────────────────────────────────────────
PARAMS['desired_events'] = ['MiddleHand', 'FixatedRest', 'LeftHand', 'RightHand']  # the events to analyze; used to name the label
PARAMS['PerformAvgRef']  = True     # average re-reference
PARAMS['AddRefChannel']  = False    # reconstruct FCz (the online reference); needs
                                    # PerformAvgRef. False -> no FCz, 35 picks not 36
PARAMS['CenterByClass']  = True     # per-class mean removal (label-dependent, so the
                                    # live loop cannot apply it). HERE the centered /
                                    # uncentered epoch variants override it per build
PARAMS['PerformCsd']     = False    # current-source-density transform
PARAMS['PerformAsr']     = False    # Artifact Subspace Reconstruction on the continuous
                                    # data, BEFORE ICA and before the band-pass. Off =
                                    # every existing result unchanged. Flipping it
                                    # rebuilds the epoch cache; set LABEL too if you
                                    # want the ASR and non-ASR trees side by side
PARAMS['asr_cutoff']     = 20       # rejection threshold, SD of the clean calibration
                                    # data. Lower = more aggressive
PARAMS['asr_max_bad_chans'] = 0.1   # max bad-channel fraction a calibration window may have
PARAMS['asr_backend']    = 'asrpy'  # 'asrpy' (euclid only) or 'meegkit'. meegkit
                                    # must be 0.1.7: newer ones need pyriemann>=0.7,
                                    # and 0.12 needs numpy 2, which breaks mne 1.6.1
PARAMS['asr_method']     = 'euclid' # 'euclid', or 'riemann' (Blum et al. 2019) which
                                    # requires asr_backend='meegkit' - asrpy accepts
                                    # 'riemann' but silently runs euclid, so we reject it
PARAMS['asr_estimator']  = 'lwf'    # meegkit only. 'riemann' REQUIRES a regularising
                                    # estimator: the average reference makes the block
                                    # covariances singular and the riemannian mean needs
                                    # positive definite input ('scm' fails outright)
PARAMS['filter_method']  = 'iir'
PARAMS['LowPass']        = 8        # band-pass low edge, Hz
PARAMS['HighPass']       = 32       # band-pass high edge, Hz
PARAMS['epoch_tmin']     = -5       # epoch crop, s relative to cue
PARAMS['epoch_tmax']     = 6
PARAMS['classifier_window_s'] = 0.2                                    # 'vote' only
PARAMS['classifier_window_e'] = 4                                      # 'vote' only
PARAMS['windowed_prediction_params'] = {'win_len': 2, 'win_step': 0.25}  # 'vote' only
PARAMS['augmentation_params']        = {'win_len': 0, 'win_step': 0.25}  # 0 = off
PARAMS['pipeline_name'] = 'ts+FGDA'

# ── read only by the CSP / FBCSP pipelines - inert while pipeline_name is ts+FGDA ─
PARAMS['n_components']       = 8
PARAMS['n_components_fbcsp'] = 8
PARAMS['filters_bands']      = [[7, 12], [12, 20], [20, 28], [28, 35]]

# ── set by the pipeline itself; assigning them here has NO effect ────────────────
#   bad_electrodes              per subject, from get_subject_bad_electrodes
#   events_trigger_dict         per subject, from epochs.event_id
#   epoch_tmins_and_maxes_grid  vestigial - read nowhere

# ── what to run ──────────────────────────────────────────────────────────────────
SUBJECTS        = None    # None = all; or ['BA', 'AEH']
RUN_DEFINITIONS = None    # None = both; or ['single'] / ['vote']
RUN_MODES       = None    # None = MODES ('c2c'); list(DIAGNOSTIC_MODES) for the
                          # seven-mode centering diagnostic
FORCE           = False   # ignore the epoch cache and rebuild
SAVE_FIGS       = True
N_PERMUTATIONS  = 200     # stage 3 only

# ─────────────────────────────────────────────────────────────────────────────────
paths = project_paths(label=LABEL, params_dict=PARAMS)
check_params(PARAMS, label=LABEL, paths=paths)
describe_params(PARAMS)

print(f"\nanalysis root : {paths.analysis_root}")
print(f"fullepoch out : {paths.out}")
print(f"shared cache  : {paths.cache}")
print()
full_epoch_status(paths=paths)

ACTIVE
  desired_events             * ['MiddleHand', 'FixatedRest', 'LeftHand', 'RightHand']    classes to decode; also names the Analysis/<label>/ tree
  Electorde_Group              36 ch: FC5, FC3, FC1 ... P8                               channel picks, and the order the classifier sees them in
  PerformAvgRef                True                                                      average re-reference
  AddRefChannel              * False                                                     reconstruct FCz (the online reference); requires PerformAvgRef
  CenterByClass                True                                                      per-class mean removal; each epoch variant overrides it per build
  PerformCsd                   False                                                     current-source-density transform
  filter_method                'iir'                                                     MNE filter method ('iir' / 'fir')
  LowPass                      8       

,subject,n_files,vote_c2c,sing_c2c
0,AEH,3,False,False
1,AN,4,False,False
2,AV,6,False,False
3,BA,3,False,False
4,EA,4,False,False
5,EE,3,False,False
6,ID,3,False,False
7,IO,3,False,False
8,LD,3,False,False
9,MN,4,False,False


## Stage 1 — decode every subject

For each subject: load both epoch variants from the shared cache, assert they are
trial-aligned, balance once (only if a `'Rest'` class is present), then run every
definition × mode cell on the **same folds** so all deltas are exactly paired.

A failing subject is reported and skipped; it never aborts the batch. Rerun the failures by
setting `SUBJECTS` in the Parameters cell.

In [3]:
out = run_full_epoch_batch(SUBJECTS, RUN_DEFINITIONS, RUN_MODES, force=FORCE,
                           save_figs=SAVE_FIGS, params_dict=PARAMS, paths=paths,
                           label=LABEL)

output label: 'MH_FR_LH_RH'  ->  Analysis\MH_FR_LH_RH\FullEpoch

[1/16] AEH
[AEH] preprocessing 3 recording(s): AEH_MI1.xdf, AEH_MI2.xdf, AEH_MI3.xdf

--- AEH_MI1.xdf ---
Creating RawArray with float64 data, n_channels=67, n_times=475014
    Range : 0 ... 475013 =      0.000 ...   950.026 secs
Ready.

###########################################################
removing subject specific bad electrodes from the raw data
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
FCz is in Electorde_Group but not in the data (AddRefChannel off?) - excluding it from the picks.

###########################################################
filtering the data
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 1e+02 Hz

IIR filter parameters
---------------------
Butterworth bandpass non-linear phase (one-pass forward) causal filter:
- Filter order 8 (forward)
- Cutoffs at 1.00, 100.00 Hz: -3.01, -3.01 dB

Filtering

found likely XDF file corruption (unpack requires a buffer of 8 bytes), scanning forward to next boundary chunk.


Creating RawArray with float64 data, n_channels=67, n_times=276265
    Range : 0 ... 276264 =      0.000 ...   552.528 secs
Ready.

###########################################################
removing subject specific bad electrodes from the raw data
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
FCz is in Electorde_Group but not in the data (AddRefChannel off?) - excluding it from the picks.

###########################################################
filtering the data
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 1e+02 Hz

IIR filter parameters
---------------------
Butterworth bandpass non-linear phase (one-pass forward) causal filter:
- Filter order 8 (forward)
- Cutoffs at 1.00, 100.00 Hz: -3.01, -3.01 dB

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

IIR filter parameters
---------------------
Butterworth bandstop non-linear phase (one-pass 

found likely XDF file corruption (unpack requires a buffer of 8 bytes), scanning forward to next boundary chunk.


Creating RawArray with float64 data, n_channels=67, n_times=276265
    Range : 0 ... 276264 =      0.000 ...   552.528 secs
Ready.

###########################################################
removing subject specific bad electrodes from the raw data
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
FCz is in Electorde_Group but not in the data (AddRefChannel off?) - excluding it from the picks.

###########################################################
filtering the data
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 1e+02 Hz

IIR filter parameters
---------------------
Butterworth bandpass non-linear phase (one-pass forward) causal filter:
- Filter order 8 (forward)
- Cutoffs at 1.00, 100.00 Hz: -3.01, -3.01 dB

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

IIR filter parameters
---------------------
Butterworth bandstop non-linear phase (one-pass 

## Stage 2 — group statistics and the summary table

Already run at the end of stage 1. Call it alone to rebuild the summary and group figures
**from disk**, without re-decoding anything.

In [4]:
summary_df, payload = summarize_full_epoch(params_dict=PARAMS, paths=paths, label=LABEL)
summary_df

Loaded 16 subjects (full-epoch) from C:\Users\gilad\New Git 3rd arm bci\3rd_arm_MI\Analysis\MH_FR_LH_RH\FullEpoch\Metrics\Group\group_full_epoch_vote_c2c.json
Loaded 16 subjects (full-epoch) from C:\Users\gilad\New Git 3rd arm bci\3rd_arm_MI\Analysis\MH_FR_LH_RH\FullEpoch\Metrics\Group\group_full_epoch_single_c2c.json
Added AEH: 10 folds | acc = 0.709 ± 0.076 (epoch 0.0–5.0s)
Added AN: 10 folds | acc = 0.836 ± 0.059 (epoch 0.0–5.0s)
Added AV: 10 folds | acc = 0.638 ± 0.070 (epoch 0.0–5.0s)
Added BA: 10 folds | acc = 0.721 ± 0.050 (epoch 0.0–5.0s)
Added EA: 10 folds | acc = 0.408 ± 0.060 (epoch 0.0–5.0s)
Added EE: 10 folds | acc = 0.561 ± 0.087 (epoch 0.0–5.0s)
Added ID: 10 folds | acc = 0.739 ± 0.051 (epoch 0.0–5.0s)
Added IO: 10 folds | acc = 0.630 ± 0.079 (epoch 0.0–5.0s)
Added LD: 10 folds | acc = 0.694 ± 0.080 (epoch 0.0–5.0s)
Added MN: 10 folds | acc = 0.695 ± 0.069 (epoch 0.0–5.0s)
Added NS: 10 folds | acc = 0.787 ± 0.054 (epoch 0.0–5.0s)
Added NZ: 10 folds | acc = 0.555 ± 0.045 

,subject,status,error,n_files,n_epochs,n_channels,classes,n_classes,chance,majority_baseline,...,acc_within_sd_single_c2c,acc_fold_min_single_c2c,acc_fold_max_single_c2c,f1_macro_single_c2c,bal_acc_single_c2c,runtime_s_single_c2c,acc_between_sd_vote_c2c,acc_between_sem_vote_c2c,acc_between_sd_single_c2c,acc_between_sem_single_c2c
0,AEH,ok,,NaN,NaN,NaN,,4,0.25,NaN,...,0.058985,0.606061,0.787879,0.666130,0.663889,NaN,NaN,NaN,NaN,NaN
1,AN,ok,,NaN,NaN,NaN,,4,0.25,NaN,...,0.030398,0.840909,0.931818,0.876272,0.872917,NaN,NaN,NaN,NaN,NaN
2,AV,ok,,NaN,NaN,NaN,,4,0.25,NaN,...,0.086673,0.571429,0.828571,0.669143,0.665400,NaN,NaN,NaN,NaN,NaN
3,BA,ok,,NaN,NaN,NaN,,4,0.25,NaN,...,0.037929,0.696970,0.787879,0.732562,0.727778,NaN,NaN,NaN,NaN,NaN
4,EA,ok,,NaN,NaN,NaN,,4,0.25,NaN,...,0.053063,0.365854,0.512195,0.449396,0.442714,NaN,NaN,NaN,NaN,NaN
5,EE,ok,,NaN,NaN,NaN,,4,0.25,NaN,...,0.060606,0.484848,0.666667,0.580189,0.584722,NaN,NaN,NaN,NaN,NaN
6,ID,ok,,NaN,NaN,NaN,,4,0.25,NaN,...,0.075656,0.636364,0.878788,0.796332,0.797222,NaN,NaN,NaN,NaN,NaN
7,IO,ok,,NaN,NaN,NaN,,4,0.25,NaN,...,0.103800,0.393939,0.727273,0.636515,0.633333,NaN,NaN,NaN,NaN,NaN
8,LD,ok,,NaN,NaN,NaN,,4,0.25,NaN,...,0.099175,0.515152,0.818182,0.682548,0.672222,NaN,NaN,NaN,NaN,NaN
9,MN,ok,,NaN,NaN,NaN,,4,0.25,NaN,...,0.054865,0.617647,0.818182,0.697824,0.695262,NaN,NaN,NaN,NaN,NaN


## Stage 3 — permutation test (opt-in, expensive)

Per-subject p-values for **one** cell of the matrix, named in the call below. `c2c`/`vote` is
the reported configuration; pass `mode='l2u'` for the leak-free deployable one, which is the
number a manuscript should be defending. Any of the seven `DIAGNOSTIC_MODES` works here without
having run the diagnostic first.

`n_permutations >= 200` is needed for a usable p-value; `Main_Experiment`'s 10 is far too few.
Budget roughly `n_permutations ×` the cost of one cell, per subject — run it once when the
numbers are final.

In [ ]:
# Uncomment to run. The cell is named here rather than in the params cell, because it
# is one square of the matrix rather than a property of the run.
#   'c2c' = the reported configuration; 'l2u' = the leak-free deployable one.
# perm = run_permutation_pass(SUBJECTS, definition='vote', mode='c2c',
#                             n_permutations=N_PERMUTATIONS,
#                             params_dict=PARAMS, paths=paths, label=LABEL)

## Reporting this in a manuscript

- Call it **offline single-trial decoding accuracy**, not "optimal" — it is this pipeline over
  this window, not an upper bound on decodability.
- The window [0, 5] s is fixed a priori by the paradigm (cue at 0, trial ends at 5). It was not
  chosen because accuracy peaked there.
- `vote` aggregates 2 s windows at 0.25 s step — ~87% overlap. It is an aggregation rule, **not**
  *n* independent votes, and must not be described as averaging away noise.
- The 4-class config runs **unbalanced** (balancing applies only when a `'Rest'` class is
  present), so accuracy sits against a per-subject `majority_baseline` rather than
  `1/n_classes`. Report **macro F1** beside it.
- Within-subject error bars are anticonservative (the ~10 folds share ~78% of their training
  trials). Quote the **between-subject** spread from the `GROUP` row.

All of this is written into `full_epoch_summary.json['caveats']` so it cannot be separated from
the numbers.